Things need to do:
1. python -m venv [venv name]
2. source [venv name]/bin/activate
3. pip install ipykernel
4. ipython kernel install --user --name=[venv name]
5. Change kernal manually in local

In [ ]:
import polars as pl
from pathlib import Path
import pandas as pd
from IPython.display import display
from openpyxl import load_workbook
# from openpyxl.utils.exceptions import InvalidFileException
import msoffcrypto
from io import BytesIO
# import numpy as np
import gc
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas

In [ ]:
# Configure Polars display settings
pl.Config.set_tbl_cols(-1)  # Limit to 10 columns
pl.Config.set_tbl_rows(5)  # Limit to 20 rows
pl.Config.set_tbl_width_chars(200)  # Set the width of columns to 25 characters
pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_hide_dtype_separator(True)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_cell_alignment("RIGHT")
pl.Config.float_precision=3

gc.collect()

In [ ]:
import warnings
warnings.filterwarnings("ignore")

### Export Excel raw to Snowflake

In [ ]:
# Define the file path and the password
file_path = Path("C:/Users/mandald/OneDrive - tredence_analytics/workFolder/OneDrive_1_10-14-2024/Enriched_Data/2885 Amplify ABM ESE One Time Enrichment_ESE_6570272 29 Oct 2024 - New Contacts Germany (with State).xlsx").resolve()
password = "sadsads"
sheet_nm = "data"

# Decrypt the file using msoffcrypto
decrypted = BytesIO()
with open(file_path, 'rb') as f:
    office_file = msoffcrypto.OfficeFile(f)
    office_file.load_key(password=password)
    office_file.decrypt(decrypted)

# Load the decrypted file into a pandas DataFrame
decrypted.seek(0)  # Reset the pointer to the beginning of the BytesIO object

polars_df = pl.read_excel(decrypted, sheet_name=sheet_nm, engine='calamine')

# # Changing all columns data type to "String"
# df = polars_df.with_columns([
#     pl.col(column).cast(pl.Utf8) for column in polars_df.columns
# ])

# df = df.rename(lambda col_nm: col_nm.lower())
# df = df.rename(lambda col_nm: col_nm.replace(' ', '_'))
# display(df)

In [ ]:
from datetime import date
polars_df = polars_df.with_columns(pl.lit(date(2024, 10, 29)).dt.strftime("%Y-%m-%d").alias("ENRICHED_DATE") )
display(polars_df)

# Convert Polars DataFrame to Pandas DataFrame
pandas_df = polars_df.to_pandas()

In [ ]:
# Snowflake connection details
account = 'tredence_analytics.us-east-1'
user = ''
password = ''
database = 'PROD'
schema = 'MQA_AMPLIFY_ABM'
warehouse = 'BSM_QA_WAREHOUSE_MEDIUM'
role = 'BSM_DEVELOPER'

# Establish connection
ctx = snowflake.connector.connect(
            account=account,
            user=user,
            password=password,
            warehouse=warehouse,
            DATABASE=database,   
            SCHEMA=schema,      
            role=role)

table_name = 'CONTACT_ENRICHED_P2'

# Truncate the table if it exists
truncate_query = f""" TRUNCATE TABLE IF EXISTS {database}.{schema}.{table_name} """

cursr = ctx.cursor()

try:
    # Execute the TRUNCATE TABLE command
    cursr.execute(truncate_query)
    print(f"{"*" * 10} Table {table_name} TRUNCATE successfully.")
except Exception as e:
    print(f"{"*" * 10} Error: TRUNCATE table {table_name}: {e}")
    exit()



# Write the DataFrame to Snowflake
success, nchunks, nrows, _ = write_pandas(conn=ctx, df=pandas_df, table_name=table_name, database=database, schema=schema, overwrite=True, auto_create_table=True)

# Close the connection
ctx.close()

# Check if the upload was successful
if success:
    print(f"{"*" * 10} {success} uploaded {nrows} rows in {nchunks} chunks.")
else:
    print(f"{"*" * 10} Failed to upload data.")